# Experiment E7: QLoRA Fine-Tuning for Indonesian Regulatory Question Router

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/robitalhazmi/easypasal/blob/main/notebooks/e7_qlora_training.ipynb)

> **Project:** EasyPasal — Citation-Grounded RAG for Indonesian Financial Regulations (OJK & Bank Indonesia)
> **Hardware Requirement:** Free GPU (Google Colab NVIDIA Tesla T4 16GB or Kaggle P100/T4)
> **Task:** Parameter-efficient 4-bit QLoRA fine-tuning of `Qwen/Qwen2.5-0.5B-Instruct` as a multi-task question router (`theme` + `is_answerable` + `is_injection`).

---
### 🛡️ Honest-Claims Protocol Notice (Plan §7 & §13)
This experiment fine-tunes a **narrow, lightweight question router and guardrail classifier** (<25ms latency) to fast-path unanswerable queries and prompt injections before expensive retrieval. It is **not** a general-purpose model fine-tuned on commercial legal drafting.

## 1. Verify GPU Hardware Acceleration

In [1]:
!nvidia-smi
import torch

print("PyTorch CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(
        "Device Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB",
    )

Sun Oct 25 05:56:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

In [2]:
!pip install -q peft trl bitsandbytes transformers accelerate datasets pyyaml pydantic

## 3. Clone Repository & Setup Environment

In [3]:
import os
import sys

if not os.path.exists("easypasal") and not os.path.exists("src"):
    !git clone https://github.com/robitalhazmi/easypasal.git
    %cd easypasal
elif os.path.exists("easypasal"):
    %cd easypasal

sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("src"))

Cloning into 'easypasal'...
remote: Enumerating objects: 1456, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 1456 (delta 74), reused 104 (delta 46), pack-reused 1205 (from 1)
Receiving objects: 100% (1456/1456), 4.42 MiB | 16.46 MiB/s, done.
Resolving deltas: 100% (836/836), done.
/content/easypasal


## 4. Prepare Router Datasets

In [4]:
from experiments.e7_qlora.dataset import build_router_dataset

counts = build_router_dataset(gold_dir="data/gold", output_dir="data/e7_router")
print("Dataset Generation Completed:", counts)

Dataset Generation Completed: {'train': 72, 'dev': 40, 'test': 66}


## 5. Execute Baseline Zero-Shot Evaluation (Before Fine-Tuning)

In [5]:
from experiments.e7_qlora.evaluate import evaluate_dataset

base_dev = evaluate_dataset("data/e7_router/dev.jsonl", mode="baseline_zero_shot")
base_test = evaluate_dataset("data/e7_router/test.jsonl", mode="baseline_zero_shot")
print(
    f"Baseline Dev Theme F1: {base_dev.theme_macro_f1:.4f} | Ans F1: {base_dev.answerability_metrics.f1:.4f} | Exact Match: {base_dev.overall_exact_match:.4f}"
)
print(
    f"Baseline Test Theme F1: {base_test.theme_macro_f1:.4f} | Ans F1: {base_test.answerability_metrics.f1:.4f} | Exact Match: {base_test.overall_exact_match:.4f}"
)

Baseline Dev Theme F1: 0.8333 | Ans F1: 0.9333 | Exact Match: 0.8500
Baseline Test Theme F1: 0.8371 | Ans F1: 0.9216 | Exact Match: 0.8333


## 6. Execute QLoRA Fine-Tuning with PEFT + TRL

In [6]:
from experiments.e7_qlora.config import QLoRAConfig
from experiments.e7_qlora.train import run_gpu_training

config = QLoRAConfig.from_yaml("experiments/e7_qlora/training_config.yaml")
print("QLoRA Configuration:", config.to_dict())

# Execute SFTTrainer on GPU
training_metrics = run_gpu_training(config)
print("Training Complete! Metrics:", training_metrics)

QLoRA Configuration: {'base_model_name': 'Qwen/Qwen2.5-0.5B-Instruct', 'output_dir': 'experiments/e7_qlora/adapter', 'train_data_path': 'data/e7_router/train.jsonl', 'dev_data_path': 'data/e7_router/dev.jsonl', 'test_data_path': 'data/e7_router/test.jsonl', 'load_in_4bit': True, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'bnb_4bit_compute_dtype': 'float16', 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'learning_rate': 0.0002, 'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.1, 'num_train_epochs': 3, 'per_device_train_batch_size': 4, 'per_device_eval_batch_size': 4, 'gradient_accumulation_steps': 4, 'max_seq_length': 512, 'weight_decay': 0.01, 'logging_steps': 5, 'eval_strategy': 'steps', 'eval_steps': 25, 'save_strategy': 'steps', 'save_steps': 25, 'save_total_limit': 2, 'seed': 42}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Tokenizing train dataset:   0%|          | 0/72 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/72 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
15,0.784217,0.735074,1.159714,0.885535,55974.000000


Training Complete! Metrics: {'train_runtime': 35.0529, 'train_samples_per_second': 6.162, 'train_steps_per_second': 0.428, 'total_flos': 127821536440320.0, 'train_loss': 1.8133580207824707, 'epoch': 3.0}


## 7. Evaluate Fine-Tuned Adapter & Run Category Regression Audit

In [7]:
from experiments.e7_qlora.evaluate import print_comparison_table, run_full_comparison

report = run_full_comparison(
    dev_path="data/e7_router/dev.jsonl",
    test_path="data/e7_router/test.jsonl",
    output_file="results/e7_qlora_evaluation.json",
)
print_comparison_table(report)


EXPERIMENT E7: QLoRA ROUTER & GUARDRAIL CLASSIFIER EVALUATION REPORT

--- DEV SET BENCHMARK RESULTS (N = 40) ---
Metric                           | Baseline Zero-Shot | Fine-Tuned QLoRA   | Delta       
----------------------------------------------------------------------------------------
Theme Macro-F1                   | 0.8333             | 1.0000             |      +0.1667
Theme Accuracy                   | 0.9500             | 1.0000             |      +0.0500
Answerability F1                 | 0.9333             | 1.0000             |      +0.0667
Answerability Precision          | 0.8750             | 1.0000             |      +0.1250
Answerability Recall             | 1.0000             | 1.0000             |      +0.0000
Injection Precision              | 1.0000             | 1.0000             |      +0.0000
Injection Recall                 | 1.0000             | 1.0000             |      +0.0000
Overall Exact Match              | 0.8500             | 1.0000             | 